# 03 - Prediction Model: 30-Day Readmission Risk

**Project:** Healthcare Readmission Risk Pipeline

This notebook trains and evaluates a logistic regression model for 30-day
readmission risk prediction. It covers:
- sklearn pipeline (StandardScaler + OneHotEncoder + LogisticRegression)
- Stratified 5-fold cross-validation
- ROC-AUC and Precision-Recall curves
- Bootstrap confidence intervals on key metrics
- Feature importance from model coefficients
- Export of per-patient risk scores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                              RocCurveDisplay, PrecisionRecallDisplay,
                              confusion_matrix)
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Load feature store
# df = pd.read_sql('SELECT * FROM gold.mart_patient_features', conn)

NUMERICAL_FEATURES = ['age', 'nb_hospitalisations_precedentes', 'nb_medicaments',
                       'duree_sejour', 'duree_zscore_patho', 'indice_complexite',
                       'ratio_meds_duree', 'severite_normalisee', 'taux_readmission_patho']

BINARY_FEATURES = ['flag_polymedication', 'flag_sortie_complexe', 'flag_deja_hospitalise',
                   'flag_antecedents_lourds', 'flag_long_sejour',
                   'flag_pathologie_chronique', 'flag_pathologie_critique']

CATEGORICAL_FEATURES = ['tranche_age', 'mode_sortie', 'service', 'groupe_clinique']

TARGET = 'readmission_30j'

FEATURE_COLS = NUMERICAL_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

## 1. Train/Test Split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} stays | Test: {len(X_test):,} stays')
print(f'Train readmission rate: {y_train.mean():.1%}')
print(f'Test readmission rate:  {y_test.mean():.1%}')

## 2. Sklearn Pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERICAL_FEATURES),
    ('bin', 'passthrough', BINARY_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',   # Handle class imbalance
        max_iter=1000,
        random_state=42,
        C=1.0,
        solver='lbfgs'
    ))
])

model.fit(X_train, y_train)
print('Model trained.')

## 3. Stratified Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc')
cv_f1  = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1')

print('=== CROSS-VALIDATION (5-fold stratified) ===')
print(f'AUC-ROC: {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}')
print(f'F1-Score: {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}')
print(f'AUC fold values: {np.round(cv_auc, 3)}')

## 4. Test Set Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)

print(f'Test AUC-ROC: {auc:.3f}')
print(f'Test F1-Score: {f1:.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['No readmission', 'Readmitted']))

## 5. ROC Curve and Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0], name=f'LR (AUC={auc:.3f})')
axes[0].set_title('ROC Curve')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].legend()

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[1], name='Logistic Regression')
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

## 6. Bootstrap Confidence Intervals

In [ ]:
N_BOOTSTRAP = 200
np.random.seed(42)

boot_auc, boot_f1, boot_prec, boot_rec = [], [], [], []

for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    yt, yp, yproba = y_test.iloc[idx], y_pred[idx], y_proba[idx]
    if yt.sum() > 0 and (1 - yt).sum() > 0:
        boot_auc.append(roc_auc_score(yt, yproba))
        boot_f1.append(f1_score(yt, yp, zero_division=0))

print('=== BOOTSTRAP CONFIDENCE INTERVALS (200 iterations) ===')
print(f'AUC-ROC: {np.percentile(boot_auc, 2.5):.3f} - {np.percentile(boot_auc, 97.5):.3f}')
print(f'F1-Score: {np.percentile(boot_f1, 2.5):.3f} - {np.percentile(boot_f1, 97.5):.3f}')

## 7. Feature Importance

In [ ]:
lr = model.named_steps['classifier']
ohe = model.named_steps['preprocessor'].named_transformers_['cat']
cat_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
feature_names = NUMERICAL_FEATURES + BINARY_FEATURES + list(cat_names)

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0])
}).sort_values('coefficient', key=abs, ascending=False).head(15)

print('=== TOP 15 FEATURES BY COEFFICIENT MAGNITUDE ===')
print(coef_df.round(3).to_string(index=False))

coef_df.head(15).plot(x='feature', y='coefficient', kind='barh', figsize=(10, 7))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 feature coefficients (logistic regression)')
plt.tight_layout()
plt.show()

## 8. Export Risk Scores

In [ ]:
# Predict on the full dataset
df['score_risque_modele'] = model.predict_proba(df[FEATURE_COLS])[:, 1]

df['categorie_risque'] = pd.cut(
    df['score_risque_modele'],
    bins=[0, 0.20, 0.40, 0.70, 1.0],
    labels=['Tres faible', 'Faible', 'Modere', 'Eleve'],
    right=True
)

export_cols = ['patient_id', 'pathologie', 'service', 'age', 'tranche_age',
               'score_risque_modele', 'categorie_risque', 'readmission_30j']

scores = df[export_cols].copy()
scores['score_risque_modele'] = scores['score_risque_modele'].round(4)

scores.to_csv('scores_risque_patients.csv', index=False, encoding='utf-8-sig')

print(f'Scores exported: {len(scores):,} rows')
print()
print(scores['categorie_risque'].value_counts(normalize=True).round(3))

## 9. Final Summary

In [ ]:
print('=== FINAL MODEL PERFORMANCE SUMMARY ===')
print()
print(f'Test set size:    {len(X_test):,} stays (20% stratified split)')
print(f'AUC-ROC:          {auc:.3f}')
print(f'F1-Score (class 1): {f1:.3f}')
print(f'CV AUC (5-fold):  {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}')
print()
print('Output: scores_risque_patients.csv')
print('Next step: load scores into Power BI (powerbi/dax_measures.dax)')